[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nlavadodatasci/atd-megaf-fiabilidade/blob/main/T9_manutencao_preventiva/T9_manutencao_preventiva.ipynb)

# T9 — Otimização da manutenção preventiva

**ATD — MEGAF 2026/27 · Atividade de Fiabilidade · Tema T9**

Substituir uma peça antes de falhar custa dinheiro, mas uma falha em serviço custa mais. De quantas em quantas horas compensa substituir? Este *notebook* mostra como calcular o custo por hora de uma política de substituição, com uma peça e custos **inventados**.

**Como usar:** execute as células por ordem (Shift+Enter). As primeiras secções estão resolvidas, com dados da aula ou com exemplos inventados, para mostrar o método. A última secção, **A sua vez**, lê os dados da instalação e deixa pistas: o trabalho do grupo começa aí. O capítulo deste tema no documento de apoio explica cada passo.

## 1. A política de substituição por idade

A regra: a peça é substituída quando atinge a idade $T$ (substituição **preventiva**, custo $C_p$) ou quando falha, se falhar antes (substituição **corretiva**, custo $C_f$, maior). Depois de cada substituição, a contagem recomeça.

Em cada ciclo (de uma substituição à seguinte):

- **custo esperado** $= C_p \cdot R(T) + C_f \cdot (1 - R(T))$ — com probabilidade $R(T)$ a peça chega a $T$ e paga-se $C_p$; caso contrário falha antes e paga-se $C_f$;
- **duração esperada** $= \displaystyle\int_0^T R(t)\,dt$ — a peça dura $T$ se não falhar, menos se falhar.

O custo por hora é o quociente dos dois:

$$C(T) = \frac{C_p \, R(T) + C_f \, (1 - R(T))}{\int_0^T R(t)\,dt}$$

In [1]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO = "https://raw.githubusercontent.com/nlavadodatasci/atd-megaf-fiabilidade/main/dados/"

def obter(ficheiro):
    """Vai buscar o ficheiro ao repositório. Se falhar, abre uma janela
    para o escolher no seu computador."""
    if not os.path.exists(ficheiro):
        try:
            urllib.request.urlretrieve(REPO + ficheiro, ficheiro)
        except Exception:
            from google.colab import files
            print("Escolha no seu computador o ficheiro", ficheiro)
            carregados = files.upload()
            if ficheiro not in carregados:               # veio com outro nome: renomeia
                os.rename(list(carregados)[0], ficheiro)
    return ficheiro

from scipy import integrate

peca = stats.weibull_min(2.5, scale=1000)     # Weibull com beta = 2,5 e eta = 1000 h (inventado)
Cp, Cf = 100, 1000                            # custos inventados

## 2. Um intervalo de cada vez

Para $T = 300$ h, passo a passo. O integral calcula-se numericamente com `integrate.quad`. (Em Excel: uma coluna com $t$ de 10 em 10 horas, outra com $R(t)$, e a soma das áreas dos trapézios.)

In [2]:
T = 300
R_T = peca.sf(T)
custo_ciclo = Cp * R_T + Cf * (1 - R_T)
duracao_ciclo = integrate.quad(peca.sf, 0, T)[0]

print(f"R({T}) = {R_T:.4f}")
print(f"Custo esperado por ciclo:   {custo_ciclo:.1f}")
print(f"Duração esperada do ciclo:  {duracao_ciclo:.1f} h")
print(f"Custo por hora:             {custo_ciclo / duracao_ciclo:.4f}")

R(300) = 0.9519
Custo esperado por ciclo:   143.3
Duração esperada do ciclo:  295.8 h
Custo por hora:             0.4844


A mesma conta, arrumada numa função, para dois intervalos diferentes:

In [3]:
def custo_por_hora(T, modelo, Cp, Cf):
    R_T = modelo.sf(T)
    return (Cp * R_T + Cf * (1 - R_T)) / integrate.quad(modelo.sf, 0, T)[0]

for T in [300, 1200]:
    print(f"T = {T:5d} h   R(T) = {peca.sf(T):.3f}   custo por hora = {custo_por_hora(T, peca, Cp, Cf):.4f}")

T =   300 h   R(T) = 0.952   custo por hora = 0.4844
T =  1200 h   R(T) = 0.207   custo por hora = 0.9717


Substituir muito cedo sai caro porque se pagam muitas substituições preventivas; substituir muito tarde sai caro porque a peça falha quase sempre antes. Algures entre os dois pode haver um intervalo melhor.

## 3. O termo de comparação: não fazer manutenção preventiva

Sem substituição preventiva, a peça funciona sempre até falhar: paga-se $C_f$ de cada vez, e cada ciclo dura em média o tempo médio até à falha (`modelo.mean()`).

In [4]:
print(f"Tempo médio até à falha: {peca.mean():.0f} h")
print(f"Custo por hora sem manutenção preventiva: {Cf / peca.mean():.4f}")

Tempo médio até à falha: 887 h
Custo por hora sem manutenção preventiva: 1.1271


## 4. A sua vez

Pistas:

1. Ajuste uma Weibull aos tempos do controlador (folha `ensaios_vida`) e leia os custos na folha `custos`.
2. Calcule o custo por hora para muitos valores de $T$ (por exemplo, `np.arange(200, 8000, 100)`) e desenhe a curva. Onde está o mínimo? (`np.argmin` dá a posição do menor valor.)
3. Compare o custo mínimo com o custo de não fazer manutenção preventiva. Quanto se poupa?
4. A curva é plana ou aguda perto do mínimo? O que significa isso na prática, se for mais conveniente substituir numa paragem programada?
5. Qual é a fiabilidade do controlador no intervalo ótimo, $R(T^*)$? O intervalo de menor custo é também o de maior fiabilidade?
6. Repita tudo para o gerador (use os custos do controlador, se a folha não os tiver). A curva tem mínimo? Relacione com o valor de $\beta$ e com a taxa de falha (tema T1).
7. Análise de sensibilidade: como muda o intervalo ótimo se a razão $C_f / C_p$ duplicar? E se $\beta$ estiver mal estimado?
8. Se o controlador for substituído preventivamente a cada $T^*$ horas, o que acontece à fiabilidade da instalação para a missão de 2000 h? (Pense: uma peça substituída há pouco é uma peça nova.)

In [5]:
ensaios = pd.read_excel(obter("dados_fiabilidade.xlsx"), sheet_name="ensaios_vida")
custos = pd.read_excel("dados_fiabilidade.xlsx", sheet_name="custos")
print(custos.iloc[:5, [0, 2, 3]])

tc = ensaios[ensaios["componente"] == "Controlador"]["tempo_ate_falha_h"].to_numpy()
# beta, _, eta = stats.weibull_min.fit(tc, floc=0)
# controlador = stats.weibull_min(beta, scale=eta)
# Ts = np.arange(200, 8000, 100)
# custos_h = [custo_por_hora(T, controlador, ..., ...) for T in Ts]

      componente  custo_substituicao_preventiva_eur  \
0    Interruptor                                NaN   
1  Transformador                                NaN   
2           QGBT                                NaN   
3        Gerador                                NaN   
4    Controlador                             2000.0   

   custo_falha_em_servico_eur  
0                         NaN  
1                         NaN  
2                         NaN  
3                         NaN  
4                     15000.0  
